# Évaluer une classification : métriques, seuils et validation

**Objectifs observables (90–120 min).** À la fin, vous pourrez calculer précision, rappel et F1 à partir d'une matrice de confusion, tracer comment un seuil déplace les faux positifs et les faux négatifs, choisir une métrique en fonction d'un coût concret, et distinguer validation et test final.

**Prérequis :** split stratifié, régression logistique, pipeline scikit-learn. Les données ci-dessous sont fournies par scikit-learn à des fins pédagogiques. Elles ne conviennent pas à une décision médicale.

## 1. Une classification binaire et sa baseline

Le dataset `breast_cancer` est intégré à scikit-learn. Dans cette leçon on interprète la classe `1` telle que fournie ; ce choix est pédagogique et n’a aucune portée de diagnostic. On commence par un split stratifié.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=21)
baseline = DummyClassifier(strategy='most_frequent').fit(x_train, y_train)
model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=2_000))
model.fit(x_train, y_train)
print('Baseline accuracy :', round(accuracy_score(y_test, baseline.predict(x_test)), 3))
print('Modèle accuracy :', round(accuracy_score(y_test, model.predict(x_test)), 3))

## 2. Matrice de confusion : des comptes aux métriques

Une prédiction binaire produit quatre comptes. Lignes = classe réelle ; colonnes = classe prédite. En prenant **1 comme classe positive**, la précision est $TP/(TP+FP)$, le rappel $TP/(TP+FN)$, et $F1=2PR/(P+R)$. L'accuracy est $(TP+TN)/N$ : elle peut être trompeuse si la classe positive est rare.

**Exemple numérique.** Sur 100 observations, imaginons $TP=8$, $FP=2$, $FN=4$ et $TN=86$. L'accuracy vaut 94 %, mais le rappel vaut seulement $8/12=67$ %. Quel résultat serait plus important si rater un cas positif coûte cher ?

In [ ]:
toy_counts = {'TN': 86, 'FP': 2, 'FN': 4, 'TP': 8}
toy_precision = toy_counts['TP'] / (toy_counts['TP'] + toy_counts['FP'])
toy_recall = toy_counts['TP'] / (toy_counts['TP'] + toy_counts['FN'])
toy_f1 = 2 * toy_precision * toy_recall / (toy_precision + toy_recall)
toy_accuracy = (toy_counts['TP'] + toy_counts['TN']) / sum(toy_counts.values())
print({name: round(value, 3) for name, value in {'accuracy': toy_accuracy, 'precision': toy_precision, 'recall': toy_recall, 'F1': toy_f1}.items()})

In [ ]:
test_prediction = model.predict(x_test)
cm = confusion_matrix(y_test, test_prediction)
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(cmap='Blues', values_format='d')
plt.title('Comptes sur le test'); plt.show()
print('precision :', round(precision_score(y_test, test_prediction), 3))
print('recall    :', round(recall_score(y_test, test_prediction), 3))
print('F1        :', round(f1_score(y_test, test_prediction), 3))

## 3. Le seuil déplace les types d'erreurs

La probabilité estimée est convertie en classe par `score >= seuil`. Baisser le seuil signale davantage d'observations : en général, le nombre de faux négatifs baisse et celui des faux positifs augmente. Le compromis dépend du coût de ces erreurs.

Pour **choisir** un seuil, utilisons uniquement une portion de `x_train` comme validation. Le vrai test reste réservé au bilan final. Les courbes calculées plus bas sur le test servent seulement à décrire le modèle déjà choisi.

In [ ]:
from sklearn.base import clone

subtrain_x, validation_x, subtrain_y, validation_y = train_test_split(
    x_train, y_train, test_size=0.25, stratify=y_train, random_state=21
)
validation_model = clone(model).fit(subtrain_x, subtrain_y)
validation_scores = validation_model.predict_proba(validation_x)[:, 1]
threshold_grid = np.linspace(0.05, 0.95, 19)
fp_counts, fn_counts = [], []
for decision_threshold in threshold_grid:
    proposed = (validation_scores >= decision_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(validation_y, proposed, labels=[0, 1]).ravel()
    fp_counts.append(fp)
    fn_counts.append(fn)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(threshold_grid, fp_counts, marker='o', label='Faux positifs (FP)')
ax.plot(threshold_grid, fn_counts, marker='s', label='Faux négatifs (FN)')
ax.axvline(0.5, color='gray', linestyle=':', label='Seuil 0,5')
ax.set(xlabel='Seuil sur P(classe 1)', ylabel="Nombre d'observations", title="Compromis des erreurs sur la validation")
ax.legend(); ax.grid(alpha=0.2)
plt.show()
print('Le seuil approprié dépend du coût réel de FP et FN ; aucun seuil clinique n’est proposé.')

In [ ]:
probability = model.predict_proba(x_test)[:, 1]
for threshold in [0.3, 0.5, 0.7]:
    prediction_at_threshold = (probability >= threshold).astype(int)
    print(f'seuil={threshold:.1f} precision={precision_score(y_test, prediction_at_threshold):.3f} recall={recall_score(y_test, prediction_at_threshold):.3f}')
precision_values, recall_values, thresholds = precision_recall_curve(y_test, probability)
plt.plot(recall_values, precision_values)
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Courbe precision–recall (split de démonstration)')
plt.grid(alpha=0.2); plt.show()
print('Average precision :', round(average_precision_score(y_test, probability), 3))

## 4. Validation croisée et variabilité

La validation croisée répète entraînement et validation sur plusieurs partitions de `x_train`; la pipeline complète est réajustée dans chaque fold. Cela empêche l'imputation et le scaling d'apprendre sur les observations de validation. L'écart entre folds décrit une partie de l'instabilité, mais cinq folds issus du même jeu ne représentent pas toutes les populations futures. Une comparaison statistique sérieuse utilise les mêmes folds pour les deux modèles et tient compte de cette dépendance.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)
fold_scores = cross_val_score(model, x_train, y_train, cv=cv, scoring='average_precision')
print('AP par fold :', fold_scores.round(3))
print('Moyenne ± écart-type :', round(fold_scores.mean(), 3), '±', round(fold_scores.std(), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(range(1, len(fold_scores) + 1), fold_scores, color='#346b82')
ax.axhline(fold_scores.mean(), color='#a74728', linestyle='--', label=f'Moyenne = {fold_scores.mean():.3f}')
ax.set(xlabel='Fold de validation', ylabel='Average precision', ylim=(0, 1.05), title='Variabilité entre partitions du train')
ax.legend(); plt.show()

## 5. Examiner les erreurs et les limites

Une métrique résume un groupe d’exemples. Avant d’agir, examiner cas mal classés, calibrage, provenance et représentativité des données. Les mesures du même dataset peuvent être dépendantes ; une stratification simple ne corrige pas toutes les formes de dépendance. ROC-AUC ou PR-AUC évaluent classement, pas le coût réel ni calibration.

In [ ]:
error_table = x_test.copy()
error_table['actual'] = y_test
error_table['predicted'] = test_prediction
error_table['probability_class_1'] = probability
misclassified = error_table[error_table['actual'] != error_table['predicted']]
print('Nombre d’erreurs :', len(misclassified))
misclassified[['actual', 'predicted', 'probability_class_1']].head()

## À retenir, limites et pratique

- Choisir la classe positive explicitement et relier chaque métrique au type d'erreur qu'elle compte.
- Fixer les transformations et les paramètres sur train/validation ; évaluer ensuite une fois sur test.
- Une courbe PR mesure un classement, sans fixer à elle seule un coût d'erreur acceptable.
- Si plusieurs lignes viennent d'une même personne ou d'une série temporelle, adapter le split par groupe ou par temps.

**Exercices.** Refaire à la main le calcul numérique. Sur une validation seulement, comparer deux seuils avec un coût inventé `5×FN + FP`; expliquer pourquoi ce coût reflète une application précise. Voir [`05_evaluation.ipynb`](../../exercises/intermediate/05_evaluation.ipynb).